# Hypothesis 2: Gender Disparity in Skilled Emigrant Composition

This notebook performs the paired hypothesis test using yearly female and male skilled emigration percentages.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ALPHA = 0.05
DATA_FILE = Path.cwd() / "SriLanka_Migration_Dinura_Chanupa.csv"
REQUIRED_COLS = ["year", "female_skilled_pct_annual", "male_skilled_pct_annual"]

def fmt_p(p_value: float) -> str:
    if p_value < 0.001:
        return "< 0.001"
    return f"= {p_value:.3f}"

def detect_outliers_iqr(values: np.ndarray) -> int:
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((values < lower) | (values > upper)).sum())

In [ ]:
df = pd.read_csv(DATA_FILE)
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data = df[REQUIRED_COLS].copy()
for col in REQUIRED_COLS:
    data[col] = pd.to_numeric(data[col], errors="coerce")

if data.isna().any().any():
    raise ValueError("Detected missing or non-numeric values in required columns")
if data["year"].duplicated().any():
    raise ValueError("Year duplicates detected; data is not uniquely paired")

data = data.sort_values("year")
year = data["year"].to_numpy(dtype=int)
female = data["female_skilled_pct_annual"].to_numpy(dtype=float)
male = data["male_skilled_pct_annual"].to_numpy(dtype=float)
diff = female - male
n = len(diff)

female_mean = float(np.mean(female))
female_sd = float(np.std(female, ddof=1))
male_mean = float(np.mean(male))
male_sd = float(np.std(male, ddof=1))
mean_diff = float(np.mean(diff))
sd_diff = float(np.std(diff, ddof=1))

shapiro_w, shapiro_p = stats.shapiro(diff)
outlier_count = detect_outliers_iqr(diff)

In [ ]:
if shapiro_p >= ALPHA:
    test_name = "Paired t-test (one-sided: female > male)"
    test_key = "paired_t_test"
    t_res = stats.ttest_rel(female, male, alternative="greater")
    test_stat = float(t_res.statistic)
    p_value = float(t_res.pvalue)
    dfree = n - 1
    se_diff = sd_diff / math.sqrt(n)
    t_crit = stats.t.ppf(0.975, dfree)
    ci_low = float(mean_diff - t_crit * se_diff)
    ci_high = float(mean_diff + t_crit * se_diff)
    effect_label = "Cohen dz"
    effect_size = float(mean_diff / sd_diff) if sd_diff > 0 else float("inf")
else:
    test_name = "Wilcoxon signed-rank test (one-sided: female > male)"
    test_key = "wilcoxon"
    w_res = stats.wilcoxon(female, male, alternative="greater", zero_method="wilcox")
    test_stat = float(w_res.statistic)
    p_value = float(w_res.pvalue)
    dfree = None
    ci_low, ci_high = None, None

    non_zero_diff = diff[diff != 0]
    abs_ranks = stats.rankdata(np.abs(non_zero_diff))
    w_plus = float(abs_ranks[non_zero_diff > 0].sum())
    w_minus = float(abs_ranks[non_zero_diff < 0].sum())
    denom = len(non_zero_diff) * (len(non_zero_diff) + 1) / 2
    effect_label = "Rank-biserial correlation"
    effect_size = float((w_plus - w_minus) / denom) if denom > 0 else float("nan")

decision = "Reject H0" if p_value < ALPHA else "Fail to reject H0"

print(f"Rows analyzed: {n}")
print(f"Assumption check (Shapiro-Wilk): W = {shapiro_w:.3f}, p {fmt_p(float(shapiro_p))}")
print(f"IQR outliers in paired differences: {outlier_count}")
print(f"Selected test: {test_name}")
if test_key == "paired_t_test":
    print(f"t({dfree}) = {test_stat:.3f}, p {fmt_p(p_value)}")
    print(f"95% CI for mean difference: [{ci_low:.2f}, {ci_high:.2f}]")
else:
    print(f"W = {test_stat:.3f}, p {fmt_p(p_value)}")
print(f"{effect_label}: {effect_size:.3f}")
print()
print(decision)
if p_value < ALPHA:
    print("Conclusion: Female skilled emigration share is significantly higher than male share.")
else:
    print("Conclusion: No statistically significant evidence that female share is higher.")

In [ ]:
plt.figure(figsize=(10, 5.5))
plt.plot(year, diff, color="#0b6e4f", marker="o", linewidth=2)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Figure 01(Gender Skilled-Share Gap Over Time: Female - Male, 1994-2025)")
plt.xlabel("Year")
plt.ylabel("Female - Male Skilled Share (percentage points)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
for i in range(n):
    plt.plot([0, 1], [female[i], male[i]], color="#1f77b4", alpha=0.35, linewidth=1)

plt.scatter(np.zeros(n), female, color="#d62728", label="Female skilled %", s=28)
plt.scatter(np.ones(n), male, color="#2ca02c", label="Male skilled %", s=28)
plt.xticks([0, 1], ["Female", "Male"])
plt.title("Figure 02(Paired Female vs Male Skilled Emigration Rates by Year)")
plt.ylabel("Skilled Emigration Share (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "n",
        "female_mean",
        "male_mean",
        "mean_diff_female_minus_male",
        "shapiro_w",
        "shapiro_p",
        "test_stat",
        "p_value",
        "effect_size"
    ],
    "value": [
        n,
        female_mean,
        male_mean,
        mean_diff,
        float(shapiro_w),
        float(shapiro_p),
        test_stat,
        p_value,
        effect_size
    ]
})
summary